<a href="https://colab.research.google.com/github/javier-jaime/PoC-Purgatory/blob/main/LLM_Driven_Ontology_Construction_for_Enterprise_Knowledge_Graphs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import necessary libraries, uncomment if required
!pip install rdflib
!pip install pyvis

from google.colab import userdata

from pydantic import BaseModel, Field
from google import genai
import dotenv
from google.genai import types

from rdflib import Graph, URIRef, Literal, Namespace, RDFS, RDF, OWL, XSD
from pyvis.network import Network
import uuid
import json
import os
import openai
from sklearn.metrics import precision_recall_fscore_support

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.9 MB/s eta 0:00:00


In [ ]:
class OntologyClass(BaseModel):
  id: int = Field(..., description="Unique Identifier of the class")
  name: str = Field(..., description="Name of the class")
  description: str = Field(..., description="Description of class/What the class is about/represent")

class OntologyProperty(BaseModel):
  id: int = Field(..., description="Unique Identifier of the property")
  name: str = Field(..., description="Name of the property (e.g., 'hasAccessTo', 'handles')")
  domain: str = Field(..., description="Source Class the property applies to")
  range: str = Field(..., description="Target Class the property points to")


In [ ]:
class Ontology(BaseModel):
  classes: list[OntologyClass] = Field(..., description="Classes in the ontology")
  properties: list[OntologyProperty] = Field(..., description="Properties in the ontology")

In [ ]:
json_schema = Ontology.model_json_schema()

In [ ]:
ONTOLOGY_SYSTEM_PROMPT = """
You are an advanced AI system specialized in ontology engineering and semantic modeling.
Your expertise includes identifying entity types (classes), their relationships (properties), and the formal structure of domain knowledge.

CRITICAL DISTINCTIONS:
- CLASSES are types/categories of things (e.g., "Employee", "Database", "Policy")
- PROPERTIES are relationships or attributes that connect classes (e.g., "hasAccessTo", "manages", "isLocatedIn")

You extract the SCHEMA of knowledge, not specific instances. You identify the "templates" that describe how a domain is structured.
"""

In [ ]:
def userprompt(input: str):

    ONTOLOGY_USER_PROMPT = f"""
    Your task: Analyze the text below (delimited by triple backticks) and extract the ontological structure - the classes (entity types) and properties (relationships) that define this domain.

    CRITICAL INSTRUCTIONS:

    1. CLASS IDENTIFICATION:
      - Identify general entity TYPES, not specific instances
      - Look for: nouns that represent categories, roles, objects, or concepts
      - Include abstract concepts if they're central to the domain (e.g., 'Policy', 'Compliance', 'Risk')

    2. PROPERTY IDENTIFICATION:
      - Identify RELATIONSHIP TYPES that connect classes
      - Look for: verbs, prepositions, or verb phrases that indicate how entities relate
      - CRITICAL: Properties MUST be 1-3 words maximum using camelCase (e.g., 'hasAccessTo', 'manages', 'isPartOf')
      - For each property, identify:
        * domain: The class this property originates from (the 'subject' type)
        * range: The class this property points to (the 'object' type)

    3. NAMING CONSISTENCY:
      - Use consistent, canonical names throughout
      - Avoid acronyms unless they're the dominant form (e.g., 'CEO' is fine, but spell out ambiguous ones)

    4. DOMAIN COVERAGE:
      - Extract ALL significant entity types mentioned or implied in the text
      - Capture domain-specific terminology precisely as used in the domain
      - Include both all entities (people, systems, documents) and abstract ones (policies, processes, standards)

    5. PROPERTY DOMAIN AND RANGE:
      - domain: Which class can HAVE this property (e.g., 'Employee' can 'hasAccessTo')
      - range: Which class is the TARGET of this property (e.g., 'hasAccessTo' points to 'Database')
      - Both domain and range must reference classes you've identified
      - Be precise: 'Employee' hasAccessTo 'Database' is more specific than 'Person' hasAccessTo 'Thing'

    6. DESCRIPTION QUALITY:
      - For classes: Write clear, 1-sentence descriptions explaining what the class represents
      - Use present tense and be precise (e.g., 'A person employed by the organization' not 'People who work')


    IMPORTANT REMINDERS:
    - Think about the SCHEMA, not the DATA
    - Extract types and relationship patterns, not specific facts
    - Keep property names concise (1-3 words max)
    - Ensure all domain/range values reference classes in your classes array
    - Make descriptions informative but concise

    Text to analyze (between triple backticks):
    ```
    {input.strip()}
    ```

    Output only the JSON object with 'classes' and 'properties' arrays.
    """

    return ONTOLOGY_USER_PROMPT

In [ ]:
INPUT_1 = """
Purpose. To ensure that all employees handle company data responsibly, maintaining confidentiality, integrity, and availability in accordance with corporate governance and regulatory standards.

Scope. This policy applies to all employees, contractors, and third parties with access to company data assets, including structured databases, unstructured files, and analytics platforms.
"""

INPUT_2 = """
1. Purpose

The purpose of this policy is to establish a culture of security and confidentiality at [Company Name]. In the finance sector, the protection of Material Non-Public Information (MNPI) and Personally Identifiable Information (PII) is critical to maintaining regulatory compliance and client trust.

2. Physical Workspaces (Clean Desk)

To minimize the risk of unauthorized access or accidental disclosure of sensitive information, the following rules apply to all workstations:

    Sensitive Documents: All paper files containing client data, financial records, or internal strategy must be cleared from desks and placed in locked drawers or filing cabinets when the workstation is unoccupied.

    Removable Media: USB drives, external hard drives, and encrypted tokens must not be left unattended.

    Waste Disposal: Sensitive documents must never be placed in regular trash bins. Use the designated secure shredding consoles located on each floor.

End of Day: At the conclusion of the workday, all desks must be entirely clear of printed materials.
"""

INPUT_3 = """
1. Purpose

The primary objective of this policy is to prevent cargo-related accidents and ensure that all shipments handled by CompanyName reach their destination without damage or loss. Proper load securement is mandatory to comply with Department of Transportation (DOT) regulations and to protect the safety of our drivers and the general public.

2. Standard Operating Procedures (SOP)

Before any vehicle departs the loading dock, the following securement standards must be met:

    Weight Distribution: Loads must be balanced laterally and distributed longitudinally to maintain the vehicle’s center of gravity. Overloading a single axle is strictly prohibited.

    Tie-Down Equipment: Only company-issued, rated hardware (straps, chains, and binders) may be used. Any equipment showing signs of fraying, cracking, or "kinking" must be tagged as "Out of Service" immediately.

    Friction and Blocking: Use of dunnage, pallets, and rubber friction mats is required for heavy machinery or slick-surfaced freight to prevent sliding during transit.

3. Inspection Requirements

Drivers are responsible for the "Three-Point Inspection" process during every haul:

    Pre-Trip: Verify that the load is immobilized and the rear doors/curtains are fully locked.

    In-Transit: For hauls exceeding 50 miles, the driver must pull over at a safe location within the first 25 miles to re-check strap tension.

    Post-Trip: Report any shifting that occurred during transit to the Warehouse Manager to improve future loading patterns.
"""

In [ ]:
initial_user_prompt = userprompt(INPUT_1)
print(initial_user_prompt)


    Your task: Analyze the text below (delimited by triple backticks) and extract the ontological structure - the classes (entity types) and properties (relationships) that define this domain.

    CRITICAL INSTRUCTIONS:

    1. CLASS IDENTIFICATION:
      - Identify general entity TYPES, not specific instances
      - Look for: nouns that represent categories, roles, objects, or concepts
      - Include abstract concepts if they're central to the domain (e.g., 'Policy', 'Compliance', 'Risk')

    2. PROPERTY IDENTIFICATION:
      - Identify RELATIONSHIP TYPES that connect classes
      - Look for: verbs, prepositions, or verb phrases that indicate how entities relate
      - CRITICAL: Properties MUST be 1-3 words maximum using camelCase (e.g., 'hasAccessTo', 'manages', 'isPartOf')
      - For each property, identify:
        * domain: The class this property originates from (the 'subject' type)
        * range: The class this property points to (the 'object' type)

    3. NAMING CONS

In [ ]:
import dotenv
import os
from google.colab import userdata
from google import genai

dotenv.load_dotenv(override=True)

# Initialize the genai client with the API key from Colab userdata
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Set OPENROUTER_API_KEY from Colab userdata as an environment variable
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
def run_ontology_construction(user_prompt: str):
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        config=types.GenerateContentConfig(
            system_instruction=ONTOLOGY_SYSTEM_PROMPT,
            response_mime_type="application/json",
            response_json_schema=json_schema,
            temperature=0.0
        ),
        contents=user_prompt,
    )

    result = Ontology.model_validate_json(response.text)

    return result

In [ ]:
result = run_ontology_construction(initial_user_prompt)

In [ ]:
for class_ in result.classes:
    print(class_)

id=1 name='Employee' description='A person who works for the organization under a contract of employment.'
id=2 name='Contractor' description='An external individual or entity providing services to the organization under a specific contract.'
id=3 name='ThirdParty' description="An external entity or partner that interacts with the organization's systems or data."
id=4 name='DataAsset' description='Any information resource owned or managed by the company that has value.'
id=5 name='Policy' description='A formal document outlining rules, guidelines, and responsibilities within the organization.'
id=6 name='GovernanceStandard' description='A set of internal rules and practices used to direct and control the organization.'
id=7 name='RegulatoryStandard' description='A law, regulation, or external requirement that the organization must comply with.'
id=8 name='SecurityPrinciple' description='A core objective of information security such as confidentiality, integrity, or availability.'
id=9 

In [ ]:
for prop in result.properties:
    print(prop)

id=1 name='appliesTo' domain='Policy' range='Employee'
id=2 name='appliesTo' domain='Policy' range='Contractor'
id=3 name='appliesTo' domain='Policy' range='ThirdParty'
id=4 name='handles' domain='Employee' range='DataAsset'
id=5 name='hasAccessTo' domain='Employee' range='DataAsset'
id=6 name='hasAccessTo' domain='Contractor' range='DataAsset'
id=7 name='hasAccessTo' domain='ThirdParty' range='DataAsset'
id=8 name='maintains' domain='Employee' range='SecurityPrinciple'
id=9 name='compliesWith' domain='Employee' range='GovernanceStandard'
id=10 name='compliesWith' domain='Employee' range='RegulatoryStandard'
id=11 name='isTypeOf' domain='Database' range='DataAsset'
id=12 name='isTypeOf' domain='File' range='DataAsset'
id=13 name='isTypeOf' domain='AnalyticsPlatform' range='DataAsset'


In [ ]:
sys_prompt = """
Given a subject class and its description, return all object classes that make the statement true:
<subject> rdfs:subClassOf <object>

Example:

Subject class: Apple (A sweet or tart pome fruit grown on trees, known for its crunchy texture and diverse varieties.)

Object classes:
- Fruit (The seed-bearing structure of a flowering plant, typically fleshy and developed from the ovary.)
- Root Vegetable (An edible underground plant part, such as a tuber or taproot, that stores nutrients.)
- Leaf Vegetable (A plant grown specifically for its edible leaves and stems, often rich in vitamins.)

Output:
- Fruit (Reason: Every apple is also a fruit.)
""".strip()

In [ ]:
openai_client = openai.OpenAI(api_key=os.getenv("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")

class OpenrouterEntailment:
    def __init__(self, classes):
        self.classes = classes
    def __repr__(self):
        return str(self.classes)

def query_openrouter(user_prompt: str):

    response_ent = openai_client.chat.completions.create(
        model="anthropic/claude-opus-4.5",
        messages=[{'role': 'system', 'content': sys_prompt}, {'role': 'user', 'content': user_prompt}],
        response_format={"type": "json_schema", "json_schema": {
  "name": "class_list_with_reasons",
  "schema": {
    "type": "object",
    "properties": {
      "classes": {
        "type": "array",
        "description": "A list of classes with their associated reason.",
        "items": {
          "type": "object",
          "properties": {
            "name": {
              "type": "string",
              "description": "The name of the class."
            },
            "reason": {
              "type": "string",
              "description": "Reason for the answer."
            },
            "valid": {
              "type": "boolean",
              "description": "Whether the relationship is rdfs:subClassOf."
            }
          },
          "required": [
            "name",
            "reason",
            "valid"
          ],
          "additionalProperties": False
        }
      }
    },
    "required": [
      "classes"
    ],
    "additionalProperties": False
  },
  "strict": True
}
        },
        temperature=0.0,
    )
    return OpenrouterEntailment(json.loads(response_ent.choices[0].message.content)['classes'])

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
class Entailment(BaseModel):
  classes: list[str] = Field(...)

def build_user_prompt(class1, comparisons):
    return (f"Subject class: {class1.name} ({class1.description})\n\nObject classes:\n{'\n'.join(sorted([f'- {c}' for c in comparisons]))}")

def run_entailment(result):
    full_result_ent = dict()
    for class1 in sorted(result.classes, key=lambda x: x.name):
        # if class1.name not in ['Security Control', 'Web Proxy']:
        #     continue
        comparisons = [f"{class2.name} ({class2.description})" for class2 in sorted(result.classes, key=lambda x: x.name) if class1 != class2]
        user_prompt = build_user_prompt(class1, comparisons)
        result_ent = query_openrouter(user_prompt) # gemini_api_entailment(user_prompt)
        print((class1.name, result_ent))
        full_result_ent[class1.name] = result_ent
    return full_result_ent

In [ ]:
def ignore_text_in_parentheses(string):
    for ch in ['\n', '/', ' or ']:
        string = string.split(ch)[0]
    for ch in ['?', ',']:
        string = string.replace(ch, "")
    return string.split("(")[0].strip()

def to_camel_case(string, is_property=False):
    # remove ampersand
    for ch in ['&', '-']:
       string = string.replace(ch, "")
    # ensure all substrings are capitalised
    substrings = []
    for i, substring in enumerate(ignore_text_in_parentheses(string).split(" ")):
        if substring == "":
            continue
        if is_property and i == 0:
            substrings.append(substring[0].lower() + substring[1:])
        else:
            substrings.append(substring[0].upper() + substring[1:])
    return ''.join(substrings)


NS = Namespace("https://liberai.org/ontology/")

def turtlize(result, full_result_ent, run_id=0):

    rdf_graph = Graph()

    for class_ in result.classes:
        subject = URIRef(NS[to_camel_case(class_.name)])
        rdf_graph.add((subject, RDF.type, OWL.Class))
        rdf_graph.add((subject, RDFS.label, Literal(class_.name)))
        rdf_graph.add((subject, RDFS.comment, Literal(class_.description)))

    for prop in result.properties:
        subject = URIRef(NS[to_camel_case(prop.name, is_property=True)])
        rdf_graph.add((subject, RDF.type, OWL.ObjectProperty))
        rdf_graph.add((subject, RDFS.label, Literal(prop.name)))
        rdf_graph.add((subject, RDFS.domain, URIRef(NS[to_camel_case(prop.domain)])))
        rdf_graph.add((subject, RDFS.range, URIRef(NS[to_camel_case(prop.range)])))

    for class1, super_classes in full_result_ent.items():
        subject = URIRef(NS[to_camel_case(class1)])
        for class_ in filter(lambda x: x['valid'], super_classes.classes):
            object_ = URIRef(NS[to_camel_case(class_['name'])])
            rdf_graph.add((subject, RDFS.subClassOf, object_))

    # serialisation
    rdf_graph.serialize(destination=f"ontology_{run_id}.ttl", format="turtle")

    return rdf_graph

In [ ]:
from pyvis.network import Network
import uuid
import json
import os

ADD_OBJECT_PROPERTIES = True
ADD_DATATYPE_PROPERTIES = True
XSD_BASE = str(XSD)
BASE_URI = str(NS)

def build_network(rdf_graph, run_id=0):
    # Load ontology
    g = Graph()
    g += rdf_graph

    # Do not show...
    g.remove((None, RDFS.comment, None))
    g.remove((None, RDF.type, OWL.Class))
    g.remove((None, RDF.type, OWL.ObjectProperty))
    g.remove((None, RDF.type, OWL.DatatypeProperty))

    # Remove hierarchy relationships of 2+ levels difference
    for (s, o) in g.query("SELECT ?s ?o WHERE { ?s rdfs:subClassOf/rdfs:subClassOf ?o }"):
        g.remove((s, RDFS.subClassOf, o))

    net = Network(
        height="750px", width="100%", directed=True, notebook=False,
        bgcolor="#ffffff", font_color="#e8eaed"
    )
    options = {
    "nodes": {
        "font": {
            "size": 16,
            "color": "#e8eaed",
        },
        "color": {
        "border": "#6da3d6",
        "background": "#1b2838",
        "highlight": {
            "border": "#87b0e6",
            "background": "#3f5a7f"
        },
        "hover": {
            "border": "#87b0e6",
            "background": "#3f5a7f"
        }
        }
    },
    "edges": {
        "font": {
        "size": 14,
        "color": "#e8eaed",
        "background": "rgba(14,17,22,0.2)",  # dark halo behind text
        "strokeWidth": 3,                     # outline thickness
        "strokeColor": "#0e1116"              # outline color (dark bg)
        },
        "color": {"color": "#9aa0a6", "highlight": "#e8eaed"},
        "smooth": {"type": "continuous"}
    },
        "interaction": {"hover": True},
    "layout": {"improvedLayout": True, "randomSeed": 2},
    "physics": {
        "enabled": False,
        "solver": "repulsion",
        "repulsion": {
        "centralGravity": 0.0,
        "springLength": 120,   # ↑ more spacing
        "springConstant": 0.02,
        "nodeDistance": 320,   # ↑ min distance between nodes
        "damping": 0.09
        }
    }
    }
    net.set_options(json.dumps(options))

    CLASS_COLOR = {"background": "#3d74bf", "border": "#87b0e6"}
    # ENTITY_COLOR = {"background": "#1b2838", "border": "#6da3d6"}
    LITERAL_COLOR = {"background": "#2c2c2c", "border": "#9aa0a6"}

    for dom_, prop_, rng_ in g.query("SELECT ?dom ?prop ?rng WHERE { ?prop rdfs:domain ?dom ; rdfs:range ?rng }"):
        dom, prop, rng = str(dom_), str(prop_), str(rng_)
        net.add_node(dom, label=dom.replace(BASE_URI, ""), shape="box", color=CLASS_COLOR)
        if XSD_BASE in rng:
            if ADD_DATATYPE_PROPERTIES:
                rng_subject = rng + str(uuid.uuid4())
                net.add_node(rng_subject, label=rng.replace(XSD_BASE, "xsd:"), shape="box", color=LITERAL_COLOR)
            else:
                continue
        else:
            if ADD_OBJECT_PROPERTIES:
                rng_subject = rng
                net.add_node(rng_subject, label=rng.replace(BASE_URI, ""), shape="box", color=CLASS_COLOR)
            else:
                continue
        net.add_edge(dom, rng_subject, label=prop.replace(BASE_URI, ""), width=5)

    for s, o in g.query("SELECT ?s ?o WHERE { ?s rdfs:subClassOf ?o }"):
        net.add_node(str(s), label=s.replace(BASE_URI, ""), shape="box", color=CLASS_COLOR)
        net.add_node(str(o), label=o.replace(BASE_URI, ""), shape="box", color=CLASS_COLOR)
        net.add_edge(str(s), str(o), label="", dashes=True, width=5)

    output_file = f"ontology_{run_id}{'-with-object-properties' if ADD_OBJECT_PROPERTIES else ''}{'-with-datatype-properties' if ADD_DATATYPE_PROPERTIES else ''}.html"

    html = net.generate_html(output_file, notebook=False)

    html = html.replace('height: 750px;', 'height: 100vh;')
    html = html.replace('border: 1px solid lightgray;', 'padding: 0; border: 0;')
    html = html.replace('</style>', 'body { background-color: #ffffff; } .card { border: 0; }\n</style>')

    with open(output_file, 'w') as f:
        f.write(html)
    return output_file


In [ ]:
def run_pipeline(input_text, run_id=0):
    initial_user_prompt = userprompt(input_text)
    result = run_ontology_construction(initial_user_prompt)
    full_result_ent = run_entailment(result)
    rdf_graph = turtlize(result, full_result_ent, run_id=run_id)
    return build_network(rdf_graph, run_id=run_id)

run_pipeline(INPUT_1, run_id=1)
# run_pipeline(INPUT_2, run_id=2)
# run_pipeline(INPUT_3, run_id=3)

In [ ]:
def rindex(string, substring):
    try:
        return string.rindex(substring)
    except ValueError:
        return -1

def to_local_name(uri):
    uri = str(uri)
    return uri[max(rindex(uri, '/'), rindex(uri, '#')) + 1:]

gold_standards_raw = [
"""
Policy appliesTo BusinessPerson
BusinessPerson hasAccessTo CompanyDataAsset
Database subClassOf CompanyDataAsset
UnstructuredFile subClassOf CompanyDataAsset
AnalyticsPlatform subClassOf CompanyDataAsset
Employee handles CompanyDataAsset
ThirdParty subClassOf BusinessPerson
Contractor subClassOf BusinessPerson
Employee subClassOf BusinessPerson
Employee isInAccordanceWith CorporateGovernanceIssue
Employee isInAccordanceWith RegulatoryStandard
""",
"""
Rule appliesTo Workstation
SensitiveDocument storedAt Workstation
SensitiveDocument contains SensitiveData
MaterialNonPublicInformation subClassOf SensitiveData
PersonallyIdentifiableInformation subClassOf SensitiveData
ClientData subClassOf SensitiveData
FinancialRecord subClassOf SensitiveData
InternalStrategyDocument subClassOf SensitiveData
""",
"""
Trip completedBy Vehicle
Driver performs Trip
Driver drives Vehicle
Vehicle departs LoadingDock
Load loadedOn Vehicle
Load loadedAt LoadingDock
Driver fills InspectionReport
WarehouseManager reviews InspectionReport
"""
]


def run_evaluation(run_id):
    graph = Graph()
    graph.parse(f"ontology_{run_id}.ttl")

    pred = set()
    for prop in [RDFS.subClassOf, RDFS.domain, RDFS.range]:
        for s, p, o in graph.triples((None, prop, None)):
            pred.add(f"{to_local_name(s)} {to_local_name(p)} {to_local_name(o)}")

    display(pred)

    gold = set()
    for gold_triple in gold_standards_raw[run_id - 1].strip().split('\n'):
        s, p, o = gold_triple.split()
        if p == 'subClassOf':
            gold.add(gold_triple)
        else:
            gold.add(f"{p} domain {s}")
            gold.add(f"{p} range {o}")

    display(gold)

    # 1. universe of triples
    universe = sorted(gold | pred)

    # 2. binary labels
    y_true = [1 if t in gold else 0 for t in universe]
    y_pred = [1 if t in pred else 0 for t in universe]

    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary"
    )

    print(p, r, f1)
    return pred, gold, p, r, f1

pred, gold, p, r, f1 = run_evaluation(run_id=1)
# run_evaluation(run_id=2)
# run_evaluation(run_id=3)

In [ ]:
universe = sorted(gold | pred)

In [ ]:
def gemini_text_embeddings(texts: list[str]):
    model_name = "gemini-embedding-001"
    response = client.models.embed_content(
        model=model_name,
        contents=texts,
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
    )
    return {text: x.values for text, x in zip(texts, response.embeddings)}

embeddings = gemini_text_embeddings(universe)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_recall_fscore_support

def find_close_match(t1, group):
    for t2 in group:
        sim = cosine_similarity([embeddings[t1]], [embeddings[t2]])[0][0]
        if sim >= 0.94:
            print((round(sim, 2), t1, t2))
            return True
    return False

# 2. binary labels
y_true = [1 if find_close_match(t, gold) else 0 for t in universe]
y_pred = [1 if find_close_match(t, pred) else 0 for t in universe]

p, r, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="binary"
)

p, r, f1